<p align="center">
    <img src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Vertical-SinFondo.png" width="500"/>
</p>

<h2 align="center"><i>ITESO, Universidad Jesuita de Guadalajara</i></h2>
<h2 align="center"><i>Quantitative Finance</i></h2>
<h2 align="center"><i>Prof. Luis Carlos Alvarado Garnica</i></h2>

# Heston — Calibración de Mercado I
---
Hasta ahora resolvimos el problema **directo**: dados los parámetros, calcular precios. La calibración es el problema **inverso**: dados precios de mercado observados, encontrar los parámetros que los reproducen.

$$\text{Parámetros } (v_0,\theta,\kappa,\xi,\rho) \;\xrightarrow{\text{pricing (directo)}}\; \text{Precios}$$
$$\text{Precios de mercado} \;\xrightarrow{\text{calibración (inverso)}}\; \text{Parámetros}$$

En esta primera sesión montamos: **función de pérdida, restricciones, y la optimización en dos etapas**. Descargamos datos reales desde Yahoo Finance y hacemos una primera calibración.


## 0. Librerias

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.integrate import quad
from scipy.optimize import differential_evolution, least_squares
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# yfinance para datos reales (instalar con: pip install yfinance)
import yfinance as yf
# pip install yfinance  / pip3 install yfinance

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f9f9f9',
                     'axes.grid':True,'grid.alpha':0.35,'font.size':11,'lines.linewidth':2})
PURPLE='#534AB7'; GREEN='#1D9E75'; ORANGE='#D85A30'; AMBER='#BA7517'; BLUE='#2A7FBF'


## 1. Motor de pricing

Las mismas funciones de siempre. Añadimos la vol implicita por bisección, que necesitaremos para comparar en unidades de volatilidad.


In [ ]:
def heston_cf_j(u, j, x, v, tau, r, kappa, theta, xi, rho):
    i=1j; uj=0.5 if j==1 else -0.5; bj=kappa-rho*xi if j==1 else kappa
    dj=np.sqrt((rho*xi*i*u-bj)**2+xi**2*(u**2-2*uj*i*u))
    g2j=(bj-rho*xi*i*u+dj)/(bj-rho*xi*i*u-dj)
    Dj=((bj-rho*xi*i*u+dj)/xi**2)*((1-np.exp(dj*tau))/(1-g2j*np.exp(dj*tau)))
    Cj=r*i*u*tau+(kappa*theta/xi**2)*((bj-rho*xi*i*u+dj)*tau-2*np.log((1-g2j*np.exp(dj*tau))/(1-g2j)))
    return np.exp(Cj+Dj*v+i*u*x)

def heston_prob_j(j,x,v,tau,K,r,kappa,theta,xi,rho):
    lnK=np.log(K)
    ig=lambda u:np.real(np.exp(-1j*u*lnK)*heston_cf_j(u,j,x,v,tau,r,kappa,theta,xi,rho)/(1j*u))
    val,_=quad(ig,1e-8,200,limit=200); return 0.5+val/np.pi

def heston_call(S,K,r,tau,v0,kappa,theta,xi,rho):
    x=np.log(S)
    return (S*heston_prob_j(1,x,v0,tau,K,r,kappa,theta,xi,rho)
            - K*np.exp(-r*tau)*heston_prob_j(2,x,v0,tau,K,r,kappa,theta,xi,rho))

def bs_call(S,K,r,tau,sigma):
    d1=(np.log(S/K)+(r+0.5*sigma**2)*tau)/(sigma*np.sqrt(tau)); d2=d1-sigma*np.sqrt(tau)
    return S*norm.cdf(d1)-K*np.exp(-r*tau)*norm.cdf(d2)

def bs_implied_vol(price,S,K,r,tau,tol=1e-7):
    intr=max(S-K*np.exp(-r*tau),0.0)
    if price<=intr+1e-8: return np.nan
    lo,hi=1e-6,5.0
    for _ in range(100):
        mid=(lo+hi)/2; d=bs_call(S,K,r,tau,mid)-price
        if abs(d)<tol: return mid
        lo,hi=(mid,hi) if d<0 else (lo,mid)
    return mid


## 2. Descargar datos reales de opciones (Yahoo Finance)

Elegimos un ticker liquido (por defecto **SPY**, el ETF del S&P 500) y descargamos su cadena de opciones. Yahoo entrega, para cada vencimiento, todos los strikes con su bid/ask, volumen e interés abierto.

> **Nota sobre la ejecución:** si `yfinance` no puede conectarse (sin internet, o Yahoo bloquea), la siguiente celda genera una **superficie sintética** con parámetros conocidos, para que el notebook siempre corra. En clase, con internet, usarás datos reales.


In [ ]:
TICKER = "SPY" 
r      = 0.045       

def descargar_cadena_yahoo(ticker, n_expiries=3):
    pass


S0, raw = descargar_cadena_yahoo(TICKER, n_expiries=3)
FUENTE = f"Yahoo Finance ({TICKER})"
print(f"Descargados {len(raw)} quotes reales. Spot S0 = {S0:.2f}")

## 3. Limpieza de datos

Los quotes crudos vienen sucios. Antes de calibrar:
- **Precios mid:** $(bid+ask)/2$, guardando el spread como señal de calidad.
- **Filtrar iliquidos:** quitar strikes con volumen cero o spreads absurdos.
- **Quitar ITM profundo:** aportan poca información de volatilidad.

Nos quedamos con opciones cercanas al dinero (moneyness entre 0.85 y 1.15).


In [ ]:
# Código aquí

## 4. La función de pérdida

Calibrar es minimizar una **función de pérdida** que mide la distancia entre modelo y mercado:

$$\mathcal{L}(\Theta) = \sum_{i=1}^{N} w_i\,\bigl[\,g_{\text{modelo}}(K_i,T_i;\Theta) - g_{\text{mkt}}(K_i,T_i)\,\bigr]^2$$

Usamos error en **precio** ponderado por el inverso del spread al cuadrado: así las opciones con bid-ask apretado (más liquidas, más confiables) pesan más en el ajuste.


In [ ]:
def loss(params, market, S0, r):
    pass

# Prueba con un punto arbitrario
test = (0.04, 0.04, 1.5, 0.5, -0.5)
print("Loss en punto de prueba:", round(loss(test, market, S0, r), 4))


## 5. Restricciones de parámetros

El optimizador debe respetar el dominio del modelo:

| Parámetro | Restricción | Razón |
|-----------|-------------|-------|
| $v_0$ | $>0$ | varianza positiva |
| $\theta$ | $>0$ | varianza de largo plazo positiva |
| $\kappa$ | $>0$ | reversión (no explosión) |
| $\xi$ | $>0$ | vol-de-vol positiva |
| $\rho$ | $\in(-1,1)$ | correlación valida |

La condición de Feller la dejamos **libre** (no la imponemos): el mercado puede exigir un skew que la viole.


In [ ]:
bounds = [
    (0.005, 0.20),   # v0
    (0.005, 0.20),   # theta
    (0.10,  6.00),   # kappa
    (0.05,  1.50),   # xi
    (-0.95, 0.50),   # rho
]
nombres = ['v0','theta','kappa','xi','rho']


## 6. Optimización en dos etapas

La superficie de pérdida es **no convexa**: tiene múltiples mínimos locales y valles planos. Por eso usamos la estrategia estándar en dos etapas:

1. **Búsqueda global** (`differential_evolution`) para encontrar un mínimo local prometedor, aunque sea lento.
2. **Refinamiento local** (`least_squares`, Levenberg-Marquardt) para precisión y velocidad.


In [ ]:
#Código aquí

## 7. Evaluación del ajuste

Comparamos la sonrisa del modelo calibrado contra la observada, por vencimiento. Deben superponerse. Reportamos también el RMSE en puntos de volatilidad.


In [ ]:
fig, ax = plt.subplots(figsize=(11,6))
errores = []
for T in sorted(set(row[1] for row in market)):
    filas_T = [row for row in market if row[1]==T]
    Ks = np.array([row[0] for row in filas_T]); orden=np.argsort(Ks); Ks=Ks[orden]
    iv_mkt = np.array([bs_implied_vol(row[2],S0,row[0],r,T)*100 for row in filas_T])[orden]
    iv_mdl = []
    for K in Ks:
        pm = heston_call(S0,K,r,T,theta_fit[0],theta_fit[2],theta_fit[1],theta_fit[3],theta_fit[4])
        iv_mdl.append(bs_implied_vol(pm,S0,K,r,T)*100)
    iv_mdl=np.array(iv_mdl)
    errores.extend(iv_mdl - iv_mkt)
    l=ax.plot(Ks, iv_mkt, 'o', label=f'Mercado T={T:.2f}')[0]
    ax.plot(Ks, iv_mdl, '-', color=l.get_color(), label=f'Heston T={T:.2f}')
ax.axvline(S0, color='gray', ls=':')
ax.set_xlabel('Strike K'); ax.set_ylabel('Vol implicita (%)')
ax.set_title('Ajuste Heston vs Mercado (por vencimiento)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

rmse = np.sqrt(np.mean(np.array(errores)**2))
print(f"RMSE del ajuste: {rmse:.3f} puntos de volatilidad")
feller = 2*theta_fit[2]*theta_fit[1] >= theta_fit[3]**2
print(f"Condicion de Feller: {'CUMPLE' if feller else 'VIOLADA'} "
      f"(2*kappa*theta={2*theta_fit[2]*theta_fit[1]:.3f}, xi^2={theta_fit[3]**2:.3f})")
